# Libraries

In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import sentencepiece as spm

In [2]:
dataset_dir = os.path.join('..','datasets')
dataset_path = os.path.join(dataset_dir, 'Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)

# Data Processing

In our exploratory data analysis (EDA), we've identified the columns that we will be working with, namely, Content, and Brand. However, before training the actual model, we'll need to format the data in numbers since that's how the model can understand and learn the patterns.

We'll create a copy of the original dataframe just in case we want a reference to the original dataset, then use the copy for any modifications and processing that we'll perform.

In [3]:
news_df = df[['Brand','Content','Label']].copy()
news_df

,Brand,Content,Label
0,Inquirer,Pollution caused by traditional cooking fuel i...,Credible
1,Manila Times,Justice Secretary Vitaliano Aguirre 2nd and Ph...,Credible
2,Inquirer,President Rodrigo Duterte on Monday night desc...,Credible
3,Manila Times,THE militant fisher folk group Pambansang Laka...,Credible
4,Inquirer,Magdalo Rep. Gary Alejano is willing to lead t...,Credible
...,...,...,...
22453,Get Real Philippines,"Indeed, everybody is shocked — just shocked! —...",Not Credible
22454,Manila Times,"A TOTAL of 132,259 individuals from 28,101 fam...",Credible
22455,Adobo Chronicles,Shortly after Rod Duterte announced there will...,Not Credible
22456,Adobo Chronicles,President Barack Obama met for the first time ...,Not Credible


There are 2 things we need to address in our dataset:
- Class Imbalance - The Credible label is twice as large as the Not Credible label which may introduce label bias to the model where they predict "Credible" for majority of the dataset.

- String to Number - The model can't understand strings, so, we'll have to convert the strings to numbers, where each word or symbol is a unique number.

## Addressing the Class Imbalance

In [4]:
news_df.Label.value_counts()

Label
Credible        14802
Not Credible     7656
Name: count, dtype: int64

Let's first begin by defining the majority class and the minority class.

In [5]:
majority = news_df[news_df['Label'] == 'Credible']
minority = news_df[news_df['Label'] == 'Not Credible']
print(f'The majority has a length of {len(majority)} while the minority has {len(minority)}')
print(f'The difference between the two is {len(majority) - len(minority)}.')
print(f'The minority is about {round(len(minority)/len(majority) * 100, 2)}% of the majority')

The majority has a length of 14802 while the minority has 7656
The difference between the two is 7146.
The minority is about 51.72% of the majority


To resolve this, we can upsample the minority class to match that of the majority class, then concatenate them.

In [6]:
news_df_upsampled = pd.concat([
    majority,
    minority.sample(len(majority), random_state = 42, replace = True)
]).sample(frac = 1, random_state = 42).reset_index(drop = True) # Shuffle Dataset

news_df_upsampled.Label.value_counts()

Label
Not Credible    14802
Credible        14802
Name: count, dtype: int64

In [7]:
news_df_upsampled.head()

,Brand,Content,Label
0,Adobo Chronicles,U.S. President-elect Donald Trump has sent a v...,Not Credible
1,GRPundit,TO ALL FILIPINOS HERE & ABROAD:As you too must...,Not Credible
2,Inquirer,The Philippine Drug Enforcement Agency (PDEA) ...,Credible
3,Manila Times,Low-cost carrier Cebu Pacific said it would di...,Credible
4,GRPundit,I often wonder why Filipinos can't behave in a...,Not Credible


## Addressing String Encoding (String-to-Number Conversion)

In this section, we mainly deal with 2 things:
1. The Label encoding into numerical values
2. Brand and Content encoding into numerical values

### Label Encoding

Encoding the label is quite easy as we only need to create a dictionary of all the unique classes in our label. Our label consist only of 2 classes: Credible and Not Credible.

Considering that we are making a fake news detection model, we would want to answer the question "Is this news fake?" which would mean that a True (1) would mean it is fake, and False (0) would indicate it is not fake. We can convert using these by creating a dictionary and applying it to the dataset.

In [8]:
label_to_idx = {
    'Credible': 0,
    'Not Credible': 1
}

news_df_upsampled['Label'] = news_df_upsampled['Label'].map(label_to_idx)
news_df_upsampled.head()

,Brand,Content,Label
0,Adobo Chronicles,U.S. President-elect Donald Trump has sent a v...,1
1,GRPundit,TO ALL FILIPINOS HERE & ABROAD:As you too must...,1
2,Inquirer,The Philippine Drug Enforcement Agency (PDEA) ...,0
3,Manila Times,Low-cost carrier Cebu Pacific said it would di...,0
4,GRPundit,I often wonder why Filipinos can't behave in a...,1


### Brand and Content Encoding

Now, we have to encode the content and brand into numerical values.

For this encoding, we'll utilize the Byte-Pair Encoding (BPE) tokenizer so that we can conserve memory while retaining the most information from each character used in the strings.

However, before applying the BPE Tokenizer, we'll have to combine the Brand with the Content. We can combine this by applying the Brand as a form of header or starter to the sequence, this way, the model can recognize the first parts of the sequence as the "author". We can accomplish this by adding the Brand and Content together with a colon and space after the Brand, which would look like the following:

Brand: Content

In [9]:
news_df_upsampled['Brand and Content'] = news_df_upsampled['Brand'] + ': ' + news_df_upsampled['Content']
news_df_upsampled.head()

,Brand,Content,Label,Brand and Content
0,Adobo Chronicles,U.S. President-elect Donald Trump has sent a v...,1,Adobo Chronicles: U.S. President-elect Donald ...
1,GRPundit,TO ALL FILIPINOS HERE & ABROAD:As you too must...,1,GRPundit: TO ALL FILIPINOS HERE & ABROAD:As yo...
2,Inquirer,The Philippine Drug Enforcement Agency (PDEA) ...,0,Inquirer: The Philippine Drug Enforcement Agen...
3,Manila Times,Low-cost carrier Cebu Pacific said it would di...,0,Manila Times: Low-cost carrier Cebu Pacific sa...
4,GRPundit,I often wonder why Filipinos can't behave in a...,1,GRPundit: I often wonder why Filipinos can't b...


In applying the BPE tokenizer, we need to make sure that we only train it on the training set. So, we need to first separate the dataset into train and tests.

We're only interested in using the 'Brand and Content' and 'Label' columns, so we'll use the train_test_split() function to separate only these two columns.

In [10]:
train, test = train_test_split(
    news_df_upsampled[['Brand and Content', 'Label']], 
    test_size = 0.2,
    random_state = 42
)

In [11]:
display(train.head())
display(test.head())

,Brand and Content,Label
13271,Adobo Chronicles: While the entire Internetdom...,1
7552,Inquirer: No sex tape purportedly featuring Se...,0
11813,"Inquirer: Echoing Malacañang’s reaction, the s...",0
20905,Adobo Chronicles: The EDSA (Epifanio de los Sa...,1
15973,Adobo Chronicles: The lawyers for Janet Lim-Na...,1


,Brand and Content,Label
27664,"Get Real Philippines: So earlier today, Mocha ...",1
28750,Pinoytrending Altervista: Filipinos are known ...,1
5115,Adobo Chronicles: Everyone is so focused on th...,1
12401,Inquirer: The investigation by the Department ...,0
17123,"Adobo Chronicles: Last week, The Adobo Chronic...",1


Now that we have the split, we have to export it to a text file called corpus so that we can plug it into the sentencepiece trainer. We'll write the brand and content of our trainset in a corpus.txt file and store it under our datasets directory.

In [12]:
corpus_path = os.path.join(dataset_dir, 'corpus.txt')

with open(corpus_path, 'w', encoding = 'utf-8') as f:
    for row in train['Brand and Content']:
        f.write(row + '\n')

After creating our corpus file, we'll now train a BPE tokenizer using our own corpus.

In [13]:
vocab_size = 8000
bpe_model_path = os.path.join('..','models','bpe')

spm.SentencePieceTrainer.train(
    input = corpus_path,
    model_prefix = os.path.join(bpe_model_path, 'spm'),
    vocab_size = vocab_size,
    model_type = 'bpe',
    pad_id = 3,
)

In [14]:
tokenizer = spm.SentencePieceProcessor()
tokenizer.Load(os.path.join(bpe_model_path, 'spm.model'))

True

Now that we've trained and loaded the BPE tokenizer. We can inspect some of the assigned token ids, particularly the special tokens such as the unknown token id or the end of sequence id. We can also inspect if it has implemented our pad_id correctly.

In [15]:
print(
    f'Unknown ID: {tokenizer.unk_id()}',
    f'Begin of Sequence ID: {tokenizer.bos_id()}',
    f'End of Sequence ID: {tokenizer.eos_id()}',
    f'Pad ID: {tokenizer.pad_id()}',
    sep = '\n'
)

Unknown ID: 0
Begin of Sequence ID: 1
End of Sequence ID: 2
Pad ID: 3


It looks like our BPE tokenizer has managed to correctly register our special tokens. Next, let's try to encode or decode some sample text into numbers, just to see if its functioning well.

In [16]:
sample_text = tokenizer.encode('this is some sample text', add_bos = True, add_eos = True)
sample_text

[1, 260, 76, 546, 69, 505, 48, 3335, 2]

Looks like it did manage to encode our text, let's see if it manages to recreate our text through its decoding.

In [17]:
tokenizer.decode(sample_text)

'this is some sample text'

Our BPE tokenizer is fully functional, all that is left now is to apply this to our train and test sets. Accomplishing this is relatively simple, we just utilize panda's apply function to run each row of our feature column to be encoded using our tokenizer's encode method.

In [18]:
train['Brand and Content'] = train['Brand and Content'].apply(tokenizer.encode)
test['Brand and Content'] = test['Brand and Content'].apply(tokenizer.encode)

display(train.head())
display(test.head())

,Brand and Content,Label
13271,"[397, 398, 7972, 5154, 8, 2589, 3706, 5252, 10...",1
7552,"[542, 7972, 1242, 2722, 4, 1841, 2654, 6506, 7...",0
11813,"[542, 7972, 196, 7075, 29, 1582, 7953, 7926, 5...",0
20905,"[397, 398, 7972, 220, 4354, 137, 7964, 7934, 1...",1
15973,"[397, 398, 7972, 220, 4126, 73, 6316, 790, 795...",1


,Brand and Content,Label
27664,"[4020, 4110, 295, 7972, 2129, 1433, 1346, 7941...",1
28750,"[1716, 1715, 7972, 716, 150, 1523, 73, 64, 566...",1
5115,"[397, 398, 7972, 7419, 358, 76, 426, 6153, 65,...",1
12401,"[542, 7972, 220, 1475, 133, 8, 774, 27, 1148, ...",0
17123,"[397, 398, 7972, 7233, 926, 7941, 220, 397, 39...",1


With this, we've successfully encoded the string feature into numerical representations that our model can now understand.

# Function Definitions

However, before we move on, let's create a few functions that we can import to our source code directory so that we don't have to write all this code in the future notebooks to process our dataset (or any future datasets that may be added).

In [ ]:
def balance_binary_classes(majority: pd.DataFrame, minority: pd.DataFrame, upsample: bool = True,
                    shuffle: bool = True, random_state: int | None = None) -> pd.DataFrame:
    """Balances binary classes given the majority label dataframe and minority label dataframe.

    Args:
        majority (pd.DataFrame): Records of the dataset that has the larger label class portion
        minority (pd.DataFrame): Records of the dataset that has the smaller label class portion
        upsample (bool, optional): Determines whether the returned dataset is an upsampled dataset. Defaults to True.
        shuffle (bool, optional): Determines whether the returned dataset is shuffled after balancing. Defaults to True.
        random_state (int | None, optional): Defines the seed used for sampling. Defaults to None.

    Returns:
        pd.DataFrame: Dataframe with balanced classes.
    """
    if upsample:
        balanced = pd.concat([
            majority,
            minority.sample(
                len(majority), 
                random_state = random_state, 
                replace = True
            )
        ])
    else:
        balanced = pd.concat([
            majority.sample(
                len(minority), 
                random_state = random_state, 
                replace = True
            ),
            minority,
        ])
    
    if shuffle:
        balanced = balanced.sample(
            frac = 1, 
            random_state = random_state
        ).reset_index(drop = True)
    
    
    return balanced

In [36]:
def encode_label_texts(df: pd.DataFrame, label_column: str, mapping_dict: dict) -> pd.DataFrame:
    """Encode a label column given a mapping dictionary that encodes a given class or label to an index.

    Args:
        df (pd.DataFrame): The dataframe that contains the label column to be encoded.
        label_column (str): Name of the label column within the provided df dataframe.
        mapping_dict (dict): Class mapping encoding class to an index or number.

    Returns:
        pd.DataFrame: Dataframe with labels encoded according to mapping dictionary.
    """
    df[label_column] = df[label_column].apply(mapping_dict)
    return df

In [35]:
def train_tokenizer(doc: str, model_dir: str = '..\\models\\bpe', model_prefix: str = 'spm', 
                    vocab_size: int = 8000, model_type: str = 'bpe', pad_id: int = 3) -> None:
    """Train a SentencePieceModel Tokenizer

    Args:
        doc (str): Path to the document or corpus used to train the tokenizer.
        model_dir (str, optional): Directory to download model and vocabulary. Defaults to ''.
        model_prefix (str, optional): Prefix that spm uses as names to model and vocabulary files. Defaults to 'spm'.
        vocab_size (int, optional): Allowed size of the vocabulary that the tokenizer uses. Defaults to 8000.
        model_type (str, optional): Type of model to be trained using spm's trainer. Defaults to 'bpe'.
        pad_id (int, optional): Index of the padding token. Defaults to 3.
    """
    spm.SentencePieceTrainer.train(
        input = doc,
        model_prefix = os.path.join(model_dir, model_prefix),
        vocab_size = vocab_size,
        model_type = model_type,
        pad_id = pad_id
    )
    return

In [34]:
def encode_feature_texts(df: pd.DataFrame, feature_col: str, model_path: str | None = None,
                         dataset_dir: str = '..\\datasets', model_dir: str = '..\\models\\bpe', 
                         model_prefix: str = 'spm', vocab_size: int | None = None, 
                         model_type: str = 'bpe', pad_id: int = 3) -> pd.DataFrame:
    """Encode a feature column within a dataframe using an existing spm model. 
    If no spm model path is provided, an spm model is trained by default.

    Args:
        df (pd.DataFrame): Dataframe that contains the feature column to be encoded.
        feature_col (str): Name of the feature column within the dataframe.
        model_path (str | None, optional): Path to the trained spm model tokenizer. If None, trains an spm model tokenizer. Defaults to None.
        dataset_dir (str, optional): Directory to store the corpus if an spm model is trained. Defaults to '..\\datasets'.
        model_dir (str, optional): Directory to save the model if an spm model is trained. Defaults to '..\\models\\bpe'.
        model_prefix (str, optional): Prefix to use as a name to store the model when it is trained. Defaults to 'spm'.
        vocab_size (int | None, optional): Allowed vocabulary size for the model to use if it is trained. Defaults to None.
        model_type (str, optional): Type of tokenizer the model uses when it is trained. Defaults to 'bpe'.
        pad_id (int, optional): Pad Id used by the tokenizer when a model is trained. Defaults to 3.

    Returns:
        pd.DataFrame: Dataframe with encoded feature column using an spm model.
    """
    if model_path is None:
        corpus_path = os.path.join(dataset_dir, 'corpus.txt')
        with open(corpus_path, 'w', encoding = 'utf-8') as file:
            for row in df[feature_col]:
                file.write(row + '\n')
        
        train_tokenizer(
            doc = corpus_path,
            model_dir = model_dir,
            model_prefix = model_prefix,
            vocab_size = vocab_size,
            model_type = model_type,
            pad_id = pad_id
        )
        
        model_path = os.path.join(model_dir, model_prefix + '.model')
        
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.load(model_path)
    
    df[feature_col] = df[feature_col].apply(tokenizer.encode)
    
    return df